# Eksplorasi Data Analysis (EDA) & Pelatihan Model NLP Klasifikasi Aduan Warga
### BEDAS Lapor-AI — Sub-Proyek Machine Learning Mandiri
Referensi: `prd_model_nlp.md`, `task_model_nlp.md`

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import joblib

# Import preprocessing
sys.path.append(os.path.abspath('../src'))
try:
    from preprocessing import preprocess
except ImportError:
    from src.preprocessing import preprocess

# 1. Load Dataset (Mencari file dataset di folder ../data/ atau lokal)
dataset_paths = [
    '../data/dataset_aduan_1200.csv',
    '../data/dataset_aduan.csv',
    'dataset_aduan.csv',
    '../../dataset_aduan.csv'
]
df = None
for path in dataset_paths:
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"[✓] Dataset berhasil dimuat dari: {path}")
        break

if df is not None:
    print("Dimensi Dataset:", df.shape)
    display(df.head())
else:
    print("[!] File dataset belum ditemukan.")

In [ ]:
# 2. Distribusi Kategori (EDA)
print("Distribusi Jumlah Sampel per Kategori:")
print(df['kategori'].value_counts())

# Visualisasi Bar Chart
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='kategori', palette='Blues_r')
plt.title('Distribusi Kelas Dataset Aduan Warga')
plt.xlabel('Kategori')
plt.ylabel('Jumlah Data')
plt.tight_layout()
plt.show()

In [ ]:
# 3. Preprocessing Teks (Cleaning & Stopwords Removal)
df['teks_bersih'] = df['teks'].apply(preprocess)
df[['teks', 'teks_bersih', 'kategori']].head(10)

In [ ]:
# 4. Split Train-Test (80:20 Stratified)
X = df['teks_bersih']
y = df['kategori']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Feature Extraction TF-IDF (Unigram & Bigram)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000, sublinear_tf=True)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print("Bentuk Matriks TF-IDF Train:", X_train_vec.shape)

In [ ]:
# 5. Training & Evaluasi Model Calibrated Linear SVM
model = CalibratedClassifierCV(LinearSVC(random_state=42), cv=3)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)

print(f"Akurasi Test Set: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# 6. Visualisasi Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=model.classes_, yticklabels=model.classes_)
plt.title('Confusion Matrix - Calibrated Linear SVM')
plt.xlabel('Prediksi')
plt.ylabel('Label Sebenarnya')
plt.tight_layout()
plt.show()